<a href="https://colab.research.google.com/github/Bharath-Krishnaa/classroom-working-student-backend-development/blob/main/GLA_RetNet_PhoneBook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

In [2]:
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

PyTorch: 2.14.1+cu130
GPU: Tesla T4
CUDA: 13.0


In [4]:
%pip install -q "flash-linear-attention==0.5.2" "transformers==4.50.1" "huggingface_hub"

In [7]:
from fla.models import GLAConfig  # registers the GLA architecture with Transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

checkpoint = "m-a-p/340M-20B-GLA-pure-baseline"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(
    checkpoint,
    torch_dtype=torch.float16,
    trust_remote_code=True,
).to("cuda").eval()

print("Loaded:", checkpoint)

model.safetensors:   0%|          | 0.00/1.37G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Loaded: m-a-p/340M-20B-GLA-pure-baseline


In [8]:
prompt = "The purpose of linear attention is"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


The purpose of linear attention is to help children learn to focus on a task. It is a technique that helps children learn to pay attention to a task without distraction.
The goal of linear attention is to help children learn to


In [10]:
import torch.nn.functional as F
import fla.layers.attn as fla_attn

def sdpa_attn(q, k, v, dropout_p=0.0, softmax_scale=None, causal=False, window_size=(-1, -1), **kwargs):
    q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
    if k.shape[1] != q.shape[1]:
        rep = q.shape[1] // k.shape[1]
        k = k.repeat_interleave(rep, dim=1)
        v = v.repeat_interleave(rep, dim=1)
    tq, tk = q.shape[2], k.shape[2]
    mask, is_causal = None, False
    if causal and tq > 1:
        if tq == tk:
            is_causal = True
        else:
            mask = torch.ones(tq, tk, dtype=torch.bool, device=q.device).tril(tk - tq)
    o = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=is_causal, scale=softmax_scale)
    return o.transpose(1, 2)

def sdpa_attn_varlen(q, k, v, cu_seqlens_q, cu_seqlens_k, max_seqlen_q=None, max_seqlen_k=None,
                     dropout_p=0.0, softmax_scale=None, causal=False, window_size=(-1, -1), **kwargs):
    cq, ck = cu_seqlens_q.tolist(), cu_seqlens_k.tolist()
    outs = []
    for i in range(len(cq) - 1):
        o = sdpa_attn(q[None, cq[i]:cq[i+1]], k[None, ck[i]:ck[i+1]], v[None, ck[i]:ck[i+1]],
                      softmax_scale=softmax_scale, causal=causal)
        outs.append(o[0])
    return torch.cat(outs, 0)

fla_attn.flash_attn_func = sdpa_attn
fla_attn.flash_attn_varlen_func = sdpa_attn_varlen

In [12]:
import gc
import pandas as pd

HYB = ["hybrid-24-1", "hybrid-12-1", "hybrid-6-1", "hybrid-3-1"]
models = (["m-a-p/transformer_340M_baseline",
           "m-a-p/340M-20B-GLA-pure-baseline",
           "m-a-p/340M-20B-RetNet-pure"]
          + [f"m-a-p/340M-20B-GLA-{h}" for h in HYB]
          + [f"m-a-p/340M-20B-RetNet-{h}" for h in HYB])

tokenizer = AutoTokenizer.from_pretrained(models[1])
inputs = tokenizer("The purpose of linear attention is", return_tensors="pt").to("cuda")

rows = []
for repo in models:
    model = AutoModelForCausalLM.from_pretrained(repo, torch_dtype=torch.float16).to("cuda").eval()
    attn = getattr(model.config, "attn", None)
    n_attn = 24 if model.config.model_type == "transformer" else (len(attn["layers"]) if attn else 0)
    params = sum(p.numel() for p in model.parameters()) / 1e6

    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=20, do_sample=False)
    text = tokenizer.decode(out[0], skip_special_tokens=True)

    print(f"{repo} | {params:.1f}M | attn layers: {n_attn}\n  {text}\n")
    rows.append({"model": repo, "params_M": round(params, 1), "attn_layers": n_attn})
    del model
    gc.collect()
    torch.cuda.empty_cache()

info = pd.DataFrame(rows)
info

model.safetensors:   0%|          | 0.00/1.36G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


m-a-p/transformer_340M_baseline | 341.1M | attn layers: 24
  The purpose of linear attention is to provide a clear and concise way of thinking about the way in which we learn. It is



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


m-a-p/340M-20B-GLA-pure-baseline | 341.7M | attn layers: 0
  The purpose of linear attention is to help children learn to focus on a task. It is a technique that helps children learn to pay



... (42 outputs left out)

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


m-a-p/340M-20B-RetNet-hybrid-3-1 | 332.7M | attn layers: 8
  The purpose of linear attention is to focus on a single task or activity.
The purpose of multitasking is to focus



,model,params_M,attn_layers
0,m-a-p/transformer_340M_baseline,341.1,24
1,m-a-p/340M-20B-GLA-pure-baseline,341.7,0
2,m-a-p/340M-20B-RetNet-pure,341.1,0
3,m-a-p/340M-20B-GLA-hybrid-24-1,340.6,1
4,m-a-p/340M-20B-GLA-hybrid-12-1,339.6,2
5,m-a-p/340M-20B-GLA-hybrid-6-1,337.4,4
6,m-a-p/340M-20B-GLA-hybrid-3-1,333.1,8
7,m-a-p/340M-20B-RetNet-hybrid-24-1,340.1,1
8,m-a-p/340M-20B-RetNet-hybrid-12-1,339.0,2
9,m-a-p/340M-20B-RetNet-hybrid-6-1,336.9,4


In [ ]:
import time

@torch.no_grad()
def prefill_ms(model, L, reps=5):
    x = torch.randint(100, 32000, (1, L), device="cuda")
    for _ in range(2):
        model(x, use_cache=False)
    times = []
    for _ in range(reps):
        torch.cuda.synchronize(); t0 = time.perf_counter()
        model(x, use_cache=False)
        torch.cuda.synchronize(); times.append((time.perf_counter() - t0) * 1000)
    return sorted(times)[len(times) // 2]

@torch.no_grad()
def decode_tok_s(model, prompt_len=1024, new_tokens=64, bs=1):
    x = torch.randint(100, 32000, (bs, prompt_len), device="cuda")
    out = model(x, use_cache=True)
    cache, nxt = out.past_key_values, out.logits[:, -1:].argmax(-1)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    for _ in range(new_tokens):
        out = model(nxt, past_key_values=cache, use_cache=True)
        cache, nxt = out.past_key_values, out.logits[:, -1:].argmax(-1)
    torch.cuda.synchronize()
    return bs * new_tokens / (time.perf_counter() - t0)

eff = []
for repo in models:
    model = AutoModelForCausalLM.from_pretrained(repo, torch_dtype=torch.float16).to("cuda").eval()
    torch.cuda.reset_peak_memory_stats()
    row = {"model": repo}
    for L in [512, 1024, 2048, 4096]:
        row[f"prefill_ms_{L}"] = round(prefill_ms(model, L), 1)
    row["decode_tok_s_bs1"] = round(decode_tok_s(model, bs=1), 1)
    row["decode_tok_s_bs16"] = round(decode_tok_s(model, bs=16), 1)
    row["peak_GB"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)
    print(row)
    eff.append(row)
    del model; gc.collect(); torch.cuda.empty_cache()

eff_df = pd.DataFrame(eff)
eff_df

In [16]:
del model
gc.collect()
torch.cuda.empty_cache()

In [17]:
import time

@torch.no_grad()
def prefill_ms(model, L, reps=10):
    x = torch.randint(100, 32000, (1, L), device="cuda")
    for _ in range(2):
        model.model(x, use_cache=False)
    times = []
    for _ in range(reps):
        torch.cuda.synchronize(); t0 = time.perf_counter()
        model.model(x, use_cache=False)
        torch.cuda.synchronize(); times.append((time.perf_counter() - t0) * 1000)
    return sorted(times)[len(times) // 2]

@torch.no_grad()
def decode_tok_s(model, prompt_len=1024, new_tokens=64, bs=1):
    x = torch.randint(100, 32000, (bs, prompt_len), device="cuda")
    out = model.model(x, use_cache=True)
    cache = out.past_key_values
    nxt = model.lm_head(out.last_hidden_state[:, -1:]).argmax(-1)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    for _ in range(new_tokens):
        out = model.model(nxt, past_key_values=cache, use_cache=True)
        cache = out.past_key_values
        nxt = model.lm_head(out.last_hidden_state[:, -1:]).argmax(-1)
    torch.cuda.synchronize()
    return bs * new_tokens / (time.perf_counter() - t0)

def safe(fn):
    try:
        return round(fn(), 1)
    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        return "OOM"

eff = []
for repo in models:
    model = AutoModelForCausalLM.from_pretrained(repo, torch_dtype=torch.float16).to("cuda").eval()
    torch.cuda.reset_peak_memory_stats()
    row = {"model": repo}
    for L in [1024, 2048, 4096, 8192, 16384]:
        row[f"prefill_ms_{L}"] = safe(lambda: prefill_ms(model, L))
    row["decode_tok_s_bs1"] = safe(lambda: decode_tok_s(model, prompt_len=4096, bs=1))
    row["decode_tok_s_bs16"] = safe(lambda: decode_tok_s(model, prompt_len=4096, bs=16))
    row["peak_GB"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)
    print(row)
    eff.append(row)
    del model; gc.collect(); torch.cuda.empty_cache()

eff_df = pd.DataFrame(eff)
eff_df

{'model': 'm-a-p/transformer_340M_baseline', 'prefill_ms_1024': 55.9, 'prefill_ms_2048': 100.0, 'prefill_ms_4096': 232.0, 'prefill_ms_8192': 605.5, 'prefill_ms_16384': 1879.7, 'decode_tok_s_bs1': 24.0, 'decode_tok_s_bs16': 'OOM', 'peak_GB': 12.32}
{'model': 'm-a-p/340M-20B-GLA-pure-baseline', 'prefill_ms_1024': 60.1, 'prefill_ms_2048': 118.1, 'prefill_ms_4096': 241.6, 'prefill_ms_8192': 485.2, 'prefill_ms_16384': 971.9, 'decode_tok_s_bs1': 19.2, 'decode_tok_s_bs16': 282.4, 'peak_GB': 6.83}
{'model': 'm-a-p/340M-20B-RetNet-pure', 'prefill_ms_1024': 115.9, 'prefill_ms_2048': 223.4, 'prefill_ms_4096': 434.9, 'prefill_ms_8192': 874.0, 'prefill_ms_16384': 1749.4, 'decode_tok_s_bs1': 14.4, 'decode_tok_s_bs16': 220.5, 'peak_GB': 6.93}
{'model': 'm-a-p/340M-20B-GLA-hybrid-24-1', 'prefill_ms_1024': 58.5, 'prefill_ms_2048': 112.2, 'prefill_ms_4096': 228.2, 'prefill_ms_8192': 485.7, 'prefill_ms_16384': 1013.9, 'decode_tok_s_bs1': 20.0, 'decode_tok_s_bs16': 292.5, 'peak_GB': 6.96}
{'model': 'm-a-p

,model,prefill_ms_1024,prefill_ms_2048,prefill_ms_4096,prefill_ms_8192,prefill_ms_16384,decode_tok_s_bs1,decode_tok_s_bs16,peak_GB
0,m-a-p/transformer_340M_baseline,55.9,100.0,232.0,605.5,1879.7,24.0,OOM,12.32
1,m-a-p/340M-20B-GLA-pure-baseline,60.1,118.1,241.6,485.2,971.9,19.2,282.4,6.83
2,m-a-p/340M-20B-RetNet-pure,115.9,223.4,434.9,874.0,1749.4,14.4,220.5,6.93
3,m-a-p/340M-20B-GLA-hybrid-24-1,58.5,112.2,228.2,485.7,1013.9,20.0,292.5,6.96
4,m-a-p/340M-20B-GLA-hybrid-12-1,73.6,117.5,242.4,484.9,1051.8,17.6,328.1,7.09
5,m-a-p/340M-20B-GLA-hybrid-6-1,55.5,110.4,224.3,486.5,1133.8,19.2,330.7,7.34
6,m-a-p/340M-20B-GLA-hybrid-3-1,53.8,109.4,240.8,485.4,1277.2,21.5,268.4,7.84
7,m-a-p/340M-20B-RetNet-hybrid-24-1,112.4,218.2,424.7,854.6,1740.7,14.5,218.7,7.06
8,m-a-p/340M-20B-RetNet-hybrid-12-1,109.8,210.9,413.7,847.3,1763.4,14.4,216.2,7.18
9,m-a-p/340M-20B-RetNet-hybrid-6-1,102.7,200.4,405.4,814.0,1770.4,15.1,223.3,7.42


In [18]:
from datasets import load_dataset

text = "\n\n".join(load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")["text"])
ids = tokenizer(text, return_tensors="pt", add_special_tokens=False).input_ids[0]

@torch.no_grad()
def wikitext_ppl(model, seq_len=2048):
    nll, n = 0.0, 0
    for i in range(len(ids) // seq_len):
        x = ids[i * seq_len:(i + 1) * seq_len][None].cuda()
        logits = model(x).logits.float()
        nll += F.cross_entropy(logits[0, :-1], x[0, 1:], reduction="sum").item()
        n += seq_len - 1
    return torch.exp(torch.tensor(nll / n)).item()

ppl = []
for repo in models:
    model = AutoModelForCausalLM.from_pretrained(repo, torch_dtype=torch.float32).to("cuda").eval()
    p = wikitext_ppl(model)
    print(repo, round(p, 2))
    ppl.append({"model": repo, "wikitext_ppl": round(p, 2)})
    del model; gc.collect(); torch.cuda.empty_cache()

ppl_df = pd.DataFrame(ppl)

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

m-a-p/transformer_340M_baseline 14.61
m-a-p/340M-20B-GLA-pure-baseline 17.16
m-a-p/340M-20B-RetNet-pure 17.94
m-a-p/340M-20B-GLA-hybrid-24-1 16.03
m-a-p/340M-20B-GLA-hybrid-12-1 15.26
m-a-p/340M-20B-GLA-hybrid-6-1 15.55
m-a-p/340M-20B-GLA-hybrid-3-1 15.62
m-a-p/340M-20B-RetNet-hybrid-24-1 15.76
m-a-p/340M-20B-RetNet-hybrid-12-1 15.32
m-a-p/340M-20B-RetNet-hybrid-6-1 16.01
m-a-p/340M-20B-RetNet-hybrid-3-1 16.2


In [19]:
from google.colab import drive
drive.mount("/content/drive")
out = "/content/drive/MyDrive/linear-attn-seminar"
import os; os.makedirs(out, exist_ok=True)

info.merge(eff_df, on="model").merge(ppl_df, on="model").to_csv(f"{out}/results_340M.csv", index=False)

Mounted at /content/drive
